# E-Commerce Graph Demo

**Pipeline:** Synthetic data → Parquet → GCS → FalkorDB → Cypher

This notebook loads an **E-Commerce Graph** using direct Parquet upload — no Starburst needed.

Graph shape:
- **Customer** nodes: `customer_id`, `name`, `country`, `tier`
- **Product** nodes: `product_id`, `name`, `category`, `price`
- **SalesOrder** nodes: `order_id`, `customer_id`, `total`, `status`
- **PURCHASED** edges: Customer → Product
- **PLACED** edges: Customer → Order

In [ ]:
# ── Step 1 │ Setup & Imports ───────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────────
import requests, json, time, os
import pandas as pd
from pathlib import Path

API = "http://control-plane:8080"
H   = {"X-Username": "demo@example.com", "X-User-Role": "admin", "Content-Type": "application/json"}

def get(path):        r = requests.get(f"{API}{path}", headers=H);   r.raise_for_status(); return r.json()
def post(path, body): r = requests.post(f"{API}{path}", headers=H, json=body); r.raise_for_status(); return r.json()
def delete(path):     r = requests.delete(f"{API}{path}", headers=H); r.raise_for_status(); return r.json()

print(requests.get(f"{API}/health").json())

## Step 1 — Generate e-commerce data

In [ ]:
# ── Step 2 │ Generate Synthetic Data ──────────────────────────────────────
# Services : (local Python — no network)
# ──────────────────────────────────────────────────────────────────────────
# --- Customers ---
customers = pd.DataFrame([
    {"customer_id": 1,  "name": "Alice Chen",      "country": "USA",       "tier": "gold"},
    {"customer_id": 2,  "name": "Bob Patel",        "country": "UK",        "tier": "silver"},
    {"customer_id": 3,  "name": "Carol Santos",     "country": "Brazil",    "tier": "gold"},
    {"customer_id": 4,  "name": "David Kim",        "country": "South Korea", "tier": "bronze"},
    {"customer_id": 5,  "name": "Eva Müller",       "country": "Germany",   "tier": "silver"},
    {"customer_id": 6,  "name": "Frank Nguyen",     "country": "Vietnam",   "tier": "bronze"},
    {"customer_id": 7,  "name": "Grace Okonkwo",    "country": "Nigeria",   "tier": "gold"},
    {"customer_id": 8,  "name": "Hiro Tanaka",      "country": "Japan",     "tier": "silver"},
    {"customer_id": 9,  "name": "Isabelle Dupont",  "country": "France",    "tier": "bronze"},
    {"customer_id": 10, "name": "James O'Brien",    "country": "Ireland",   "tier": "gold"},
])

# --- Products ---
products = pd.DataFrame([
    {"product_id": 1,  "name": "Laptop Pro 15",       "category": "Electronics",  "price": 1299.99},
    {"product_id": 2,  "name": "Wireless Headphones", "category": "Electronics",  "price": 249.99},
    {"product_id": 3,  "name": "Running Shoes X10",   "category": "Sports",       "price": 119.99},
    {"product_id": 4,  "name": "Coffee Maker Deluxe", "category": "Home",         "price": 89.99},
    {"product_id": 5,  "name": "Python Cookbook",     "category": "Books",        "price": 39.99},
    {"product_id": 6,  "name": "4K Monitor 27in",     "category": "Electronics",  "price": 599.99},
    {"product_id": 7,  "name": "Yoga Mat Premium",    "category": "Sports",       "price": 59.99},
    {"product_id": 8,  "name": "Smart Watch Series 5","category": "Electronics",  "price": 399.99},
    {"product_id": 9,  "name": "Ergonomic Chair",     "category": "Home",         "price": 449.99},
    {"product_id": 10, "name": "Graph Databases Book","category": "Books",        "price": 49.99},
])

# --- SalesOrders ---
sales_orders = pd.DataFrame([
    {"order_id": 1,  "customer_id": 1,  "total": 1549.98, "status": "delivered"},
    {"order_id": 2,  "customer_id": 1,  "total": 249.99,  "status": "delivered"},
    {"order_id": 3,  "customer_id": 2,  "total": 119.99,  "status": "delivered"},
    {"order_id": 4,  "customer_id": 3,  "total": 1849.97, "status": "shipped"},
    {"order_id": 5,  "customer_id": 3,  "total": 89.99,   "status": "delivered"},
    {"order_id": 6,  "customer_id": 4,  "total": 39.99,   "status": "delivered"},
    {"order_id": 7,  "customer_id": 5,  "total": 599.99,  "status": "processing"},
    {"order_id": 8,  "customer_id": 6,  "total": 59.99,   "status": "delivered"},
    {"order_id": 9,  "customer_id": 7,  "total": 1299.99, "status": "delivered"},
    {"order_id": 10, "customer_id": 7,  "total": 449.99,  "status": "shipped"},
    {"order_id": 11, "customer_id": 8,  "total": 399.99,  "status": "delivered"},
    {"order_id": 12, "customer_id": 9,  "total": 49.99,   "status": "delivered"},
    {"order_id": 13, "customer_id": 10, "total": 1299.99, "status": "delivered"},
    {"order_id": 14, "customer_id": 10, "total": 249.99,  "status": "delivered"},
    {"order_id": 15, "customer_id": 2,  "total": 449.99,  "status": "processing"},
])

# --- PURCHASED edges: Customer → Product ---
purchased = pd.DataFrame([
    # Alice (1): Laptop, Headphones
    {"customer_id": 1,  "product_id": 1},
    {"customer_id": 1,  "product_id": 2},
    # Bob (2): Running Shoes, Ergonomic Chair
    {"customer_id": 2,  "product_id": 3},
    {"customer_id": 2,  "product_id": 9},
    # Carol (3): Laptop, Monitor, Coffee Maker
    {"customer_id": 3,  "product_id": 1},
    {"customer_id": 3,  "product_id": 6},
    {"customer_id": 3,  "product_id": 4},
    # David (4): Python Cookbook
    {"customer_id": 4,  "product_id": 5},
    # Eva (5): Monitor
    {"customer_id": 5,  "product_id": 6},
    # Frank (6): Yoga Mat
    {"customer_id": 6,  "product_id": 7},
    # Grace (7): Laptop, Ergonomic Chair
    {"customer_id": 7,  "product_id": 1},
    {"customer_id": 7,  "product_id": 9},
    # Hiro (8): Smart Watch
    {"customer_id": 8,  "product_id": 8},
    # Isabelle (9): Graph Databases Book
    {"customer_id": 9,  "product_id": 10},
    # James (10): Laptop, Headphones
    {"customer_id": 10, "product_id": 1},
    {"customer_id": 10, "product_id": 2},
])

# --- PLACED edges: Customer → SalesOrder ---
placed = sales_orders[["customer_id", "order_id"]].copy()

print(f"Customers: {len(customers)}, Products: {len(products)}, Orders: {len(sales_orders)}")
print(f"PURCHASED edges: {len(purchased)}, PLACED edges: {len(placed)}")

## Step 2 — Save to Parquet

In [ ]:
# ── Step 3 │ Save to Parquet ───────────────────────────────────────────────
# Services : (local filesystem — no network)
# ──────────────────────────────────────────────────────────────────────────
base = Path("/tmp/ecommerce-graph")
for p in ["nodes/Customer", "nodes/Product", "nodes/SalesOrder",
           "edges/PURCHASED", "edges/PLACED"]:
    (base / p).mkdir(parents=True, exist_ok=True)

customers.to_parquet(base / "nodes/Customer/data.parquet", index=False)
products.to_parquet(base  / "nodes/Product/data.parquet",  index=False)
sales_orders.to_parquet(base / "nodes/SalesOrder/data.parquet", index=False)
purchased.to_parquet(base / "edges/PURCHASED/data.parquet", index=False)
placed.to_parquet(base    / "edges/PLACED/data.parquet",    index=False)

print("Parquet files written to /tmp/ecommerce-graph/")

## Step 3 — Create Mapping

In [ ]:
# ── Step 4 │ Create Mapping ────────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────────
mapping = post("/api/mappings", {
    "name": "E-Commerce Graph",
    "description": "Customers, products, orders — synthetic e-commerce dataset",
    "node_definitions": [
        {
            "label": "Customer",
            "sql": "SELECT customer_id, name, country, tier FROM placeholder",
            "primary_key": {"name": "customer_id", "type": "INT64"},
            "properties": [
                {"name": "name",    "type": "STRING"},
                {"name": "country", "type": "STRING"},
                {"name": "tier",    "type": "STRING"},
            ]
        },
        {
            "label": "Product",
            "sql": "SELECT product_id, name, category, price FROM placeholder",
            "primary_key": {"name": "product_id", "type": "INT64"},
            "properties": [
                {"name": "name",     "type": "STRING"},
                {"name": "category", "type": "STRING"},
                {"name": "price",    "type": "DOUBLE"},
            ]
        },
        {
            "label": "SalesOrder",
            "sql": "SELECT order_id, customer_id, total, status FROM placeholder",
            "primary_key": {"name": "order_id", "type": "INT64"},
            "properties": [
                {"name": "customer_id", "type": "INT64"},
                {"name": "total",       "type": "DOUBLE"},
                {"name": "status",      "type": "STRING"},
            ]
        },
    ],
    "edge_definitions": [
        {
            "type": "PURCHASED",
            "from_node": "Customer",
            "to_node": "Product",
            "sql": "SELECT customer_id, product_id FROM placeholder",
            "from_key": "customer_id",
            "to_key": "product_id",
            "properties": []
        },
        {
            "type": "PLACED",
            "from_node": "Customer",
            "to_node": "SalesOrder",
            "sql": "SELECT customer_id, order_id FROM placeholder",
            "from_key": "customer_id",
            "to_key": "order_id",
            "properties": []
        },
    ]
})

mapping_id = mapping["data"]["id"]
print(f"Mapping created: id={mapping_id}")

## Step 4 — Create Instance

In [ ]:
# ── Step 5 │ Create Instance + Bypass Export Worker ───────────────────────
# Services : Control Plane (localhost:8081)  ·  PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────────
inst = post("/api/instances", {
    "mapping_id":    mapping_id,
    "wrapper_type":  "falkordb",
    "name":          "E-Commerce Instance",
    "ttl":           "PT4H",
    "description":   "E-commerce graph demo instance",
})
instance_id = inst["data"]["id"]
snapshot_id = inst["data"]["snapshot_id"]
print(f"Instance created: id={instance_id}")
print(f"Snapshot id:      {snapshot_id}")

# Immediately fail any pending export_jobs and mark the snapshot ready.
# This bypasses the export worker (no Starburst needed for local demo).
import psycopg2
conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="control_plane", user="control_plane", password="control_plane"
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(
        "UPDATE export_jobs SET status='failed' WHERE snapshot_id=%s AND status='pending'",
        (snapshot_id,)
    )
    print(f"  Failed {cur.rowcount} pending export_job(s)")
    cur.execute(
        "UPDATE snapshots SET status='ready' WHERE id=%s",
        (snapshot_id,)
    )
    print(f"  Snapshot {snapshot_id} marked ready")
conn.close()

## Step 5 — Upload Parquet files to GCS

In [ ]:
# GCS setup — skipped automatically in production
try:
    #         ─    ─         S    t    e    p         6         │         U    p    l    o    a    d         P    a    r    q    u    e    t         t    o         G    C    S         ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    
    #         S    e    r    v    i    c    e    s         :         l    o    c    a    l    h    o    s    t         (    l    o    c    a    l    h    o    s    t    :    4    4    4    3    )    
    #         ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    ─    
    #         U    p    l    o    a    d         p    a    r    q    u    e    t         f    i    l    e    s         d    i    r    e    c    t    l    y         t    o         f    a    k    e    -    g    c    s         v    i    a         H    T    T    P         A    P    I    
    B    U    C    K    E    T         =         "    g    r    a    p    h    -    o    l    a    p    -    l    o    c    a    l    -    d    e    v    "    
    G    C    S                        =         "    h    t    t    p    :    /    /    l    o    c    a    l    h    o    s    t    :    4    4    4    3    "    
    O    W    N    E    R              =         "    d    e    m    o    @    e    x    a    m    p    l    e    .    c    o    m    "    
    P    R    E    F    I    X         =         f    "    {    O    W    N    E    R    }    /    {    m    a    p    p    i    n    g    _    i    d    }    /    v    1    /    {    s    n    a    p    s    h    o    t    _    i    d    }    "    
    p    r    i    n    t    (    f    "    U    p    l    o    a    d    i    n    g         t    o         g    s    :    /    /    {    B    U    C    K    E    T    }    /    {    P    R    E    F    I    X    }    /    "    )    
    
    #         E    n    s    u    r    e         b    u    c    k    e    t         e    x    i    s    t    s         i    n         f    a    k    e    -    g    c    s         (    i    n    -    m    e    m    o    r    y         —         r    e    c    r    e    a    t    e    d         a    f    t    e    r         e    a    c    h         p    o    d         r    e    s    t    a    r    t    )    
    _    b    r         =         r    e    q    u    e    s    t    s    .    p    o    s    t    (    f    "    {    G    C    S    }    /    s    t    o    r    a    g    e    /    v    1    /    b    "    ,         j    s    o    n    =    {    "    n    a    m    e    "    :         B    U    C    K    E    T    }    ,    
                                                                               h    e    a    d    e    r    s    =    {    "    C    o    n    t    e    n    t    -    T    y    p    e    "    :         "    a    p    p    l    i    c    a    t    i    o    n    /    j    s    o    n    "    }    )    
    i    f         _    b    r    .    s    t    a    t    u    s    _    c    o    d    e         n    o    t         i    n         (    2    0    0    ,         2    0    1    ,         4    0    9    )    :    
                        _    b    r    .    r    a    i    s    e    _    f    o    r    _    s    t    a    t    u    s    (    )    
    
    f    i    l    e    s         =         [    
                        (    "    n    o    d    e    s    /    C    u    s    t    o    m    e    r    /    d    a    t    a    .    p    a    r    q    u    e    t    "    ,              f    "    {    P    R    E    F    I    X    }    /    n    o    d    e    s    /    C    u    s    t    o    m    e    r    /    d    a    t    a    .    p    a    r    q    u    e    t    "    )    ,    
                        (    "    n    o    d    e    s    /    P    r    o    d    u    c    t    /    d    a    t    a    .    p    a    r    q    u    e    t    "    ,                   f    "    {    P    R    E    F    I    X    }    /    n    o    d    e    s    /    P    r    o    d    u    c    t    /    d    a    t    a    .    p    a    r    q    u    e    t    "    )    ,    
                        (    "    n    o    d    e    s    /    S    a    l    e    s    O    r    d    e    r    /    d    a    t    a    .    p    a    r    q    u    e    t    "    ,                             f    "    {    P    R    E    F    I    X    }    /    n    o    d    e    s    /    S    a    l    e    s    O    r    d    e    r    /    d    a    t    a    .    p    a    r    q    u    e    t    "    )    ,    
                        (    "    e    d    g    e    s    /    P    U    R    C    H    A    S    E    D    /    d    a    t    a    .    p    a    r    q    u    e    t    "    ,         f    "    {    P    R    E    F    I    X    }    /    e    d    g    e    s    /    P    U    R    C    H    A    S    E    D    /    d    a    t    a    .    p    a    r    q    u    e    t    "    )    ,    
                        (    "    e    d    g    e    s    /    P    L    A    C    E    D    /    d    a    t    a    .    p    a    r    q    u    e    t    "    ,                        f    "    {    P    R    E    F    I    X    }    /    e    d    g    e    s    /    P    L    A    C    E    D    /    d    a    t    a    .    p    a    r    q    u    e    t    "    )    ,    
    ]    
    
    f    o    r         l    o    c    a    l    ,         r    e    m    o    t    e         i    n         f    i    l    e    s    :    
                        f    i    l    e    _    p    a    t    h         =         b    a    s    e         /         l    o    c    a    l    
                        w    i    t    h         o    p    e    n    (    f    i    l    e    _    p    a    t    h    ,         "    r    b    "    )         a    s         f    :    
                                            d    a    t    a         =         f    .    r    e    a    d    (    )    
                        u    r    l              =         f    "    {    G    C    S    }    /    u    p    l    o    a    d    /    s    t    o    r    a    g    e    /    v    1    /    b    /    {    B    U    C    K    E    T    }    /    o    ?    u    p    l    o    a    d    T    y    p    e    =    m    e    d    i    a    &    n    a    m    e    =    {    r    e    m    o    t    e    }    "    
                        r    e    s    p         =         r    e    q    u    e    s    t    s    .    p    o    s    t    (    u    r    l    ,         d    a    t    a    =    d    a    t    a    ,         h    e    a    d    e    r    s    =    {    "    C    o    n    t    e    n    t    -    T    y    p    e    "    :         "    a    p    p    l    i    c    a    t    i    o    n    /    o    c    t    e    t    -    s    t    r    e    a    m    "    }    )    
                        i    f         r    e    s    p    .    s    t    a    t    u    s    _    c    o    d    e         i    n         (    2    0    0    ,         2    0    1    )    :    
                                            p    r    i    n    t    (    f    "              O    K              {    r    e    m    o    t    e    }    "    )    
                        e    l    s    e    :    
                                            p    r    i    n    t    (    f    "              E    R    R         {    r    e    m    o    t    e    }    :         {    r    e    s    p    .    s    t    a    t    u    s    _    c    o    d    e    }         {    r    e    s    p    .    t    e    x    t    [    :    2    0    0    ]    }    "    )except Exception as _e:
    print(f'GCS setup skipped (production mode): {_e}')


## Step 6 — Poll for instance running status

In [ ]:
# ── Step 7 │ Wait for Instance ────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────────
print("Polling... (reconciliation loop runs every ~30s)")
start = time.time()
while True:
    elapsed = int(time.time() - start)
    inst    = get(f"/api/instances/{instance_id}")
    status  = inst["data"]["status"]
    print(f"  [{elapsed:3d}s] instance={status}")
    if status == "running":
        break
    if status in ("stopped", "failed"):
        raise RuntimeError(f"Instance ended with status: {status}")
    time.sleep(10)

print("\nE-commerce graph is running! FalkorDB pod ready.")

## Step 7 — Query the E-Commerce Graph

In [ ]:
# ── Step 8 │ Connect & Query Graph ────────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
inst_data = get(f"/api/instances/{instance_id}")["data"]
pod_name  = inst_data["pod_name"]
WRAPPER   = f"http://{pod_name}:8000"
print(f"Wrapper: {WRAPPER}")

def cypher(q, params=None):
    body = {"query": q}
    if params:
        body["parameters"] = params
    for attempt in range(10):
        try:
            r = requests.post(f"{WRAPPER}/query",
                              headers={"Content-Type": "application/json"}, json=body)
            r.raise_for_status()
            return r.json()["rows"]
        except Exception as e:
            if attempt < 9:
                print(f"  Wrapper not ready yet, retrying in 3s... ({attempt+1}/10)")
                time.sleep(3)
            else:
                raise

# --- Node counts ---
print("=== Graph loaded — Node counts ===")
for row in cypher("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC"):
    print(f"  {str(row[0]):20s}: {row[1]:,}")

In [ ]:
# ── Step 8a │ Cypher — Top Customers by Spend ─────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Top customers by total spend ---
print("=== Top customers by total spend ===")
results = cypher("""
    MATCH (c:Customer)-[:PLACED]->(o:SalesOrder)
    WITH c.name AS customer, c.tier AS tier, c.country AS country,
         sum(o.total) AS total_spend, count(o) AS order_count
    RETURN customer, tier, country, total_spend, order_count
    ORDER BY total_spend DESC
""")
for row in results:
    print(f"  {row[0]:22s} [{row[1]:6s}] {row[2]:15s}  ${row[3]:>8.2f}  ({row[4]} order(s))")

In [ ]:
# ── Step 8b │ Cypher — Most Popular Products ──────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Most popular products (by number of distinct purchasers) ---
print("=== Most popular products ===")
results = cypher("""
    MATCH (c:Customer)-[:PURCHASED]->(p:Product)
    WITH p.name AS product, p.category AS category, p.price AS price,
         count(c) AS purchaser_count
    RETURN product, category, price, purchaser_count
    ORDER BY purchaser_count DESC
""")
for row in results:
    print(f"  {row[0]:28s} ({row[1]:12s})  ${row[2]:>7.2f}  {row[3]} buyer(s)")

In [ ]:
# ── Step 8c │ Cypher — Co-purchaser Pairs ─────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Co-purchasers: customers who bought the same product ---
print("=== Co-purchaser pairs (customers who bought the same product) ===")
results = cypher("""
    MATCH (c1:Customer)-[:PURCHASED]->(p:Product)<-[:PURCHASED]-(c2:Customer)
    WHERE c1.customer_id < c2.customer_id
    RETURN c1.name AS customer1, c2.name AS customer2,
           p.name AS product, p.category AS category
    ORDER BY c1.name, c2.name
""")
for row in results:
    print(f"  {row[0]:22s} + {row[1]:22s}  both bought '{row[2]}' ({row[3]})")

In [ ]:
# ── Step 8d │ Cypher — Revenue by Category ────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Revenue by product category ---
print("=== Revenue by product category (from delivered orders only) ===")
results = cypher("""
    MATCH (c:Customer)-[:PLACED]->(o:SalesOrder)
    WHERE o.status = 'delivered'
    MATCH (c)-[:PURCHASED]->(p:Product)
    WITH p.category AS category,
         sum(p.price) AS category_revenue,
         count(DISTINCT c) AS unique_customers
    RETURN category, category_revenue, unique_customers
    ORDER BY category_revenue DESC
""")
for row in results:
    print(f"  {str(row[0]):15s}  revenue=${row[1]:>8.2f}  unique_customers={row[2]}")

## Step 9 — Visualise (PyVis)

In [ ]:
# ── Step 9 │ Visualise Graph (PyVis) ──────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  PyVis (local)
# ──────────────────────────────────────────────────────────────────────────
from pyvis.network import Network
from IPython.display import IFrame

net = Network(height="600px", width="100%", bgcolor="#1a1a2e", font_color="white", notebook=True)

added_nodes = set()

# PURCHASED edges: Customer → Product
rows = cypher("""
    MATCH (c:Customer)-[:PURCHASED]->(p:Product)
    RETURN c.customer_id, c.name, c.tier, p.product_id, p.name AS pname, p.category
""")

for row in rows:
    c_id, c_name, c_tier, p_id, p_name, p_category = row

    c_key = f"customer_{c_id}"
    if c_key not in added_nodes:
        net.add_node(c_key, label=c_name, color="#e94560", shape="dot", size=25)
        added_nodes.add(c_key)

    p_key = f"product_{p_id}"
    if p_key not in added_nodes:
        net.add_node(p_key, label=p_name, color="#0f3460", shape="star", size=20)
        added_nodes.add(p_key)

    net.add_edge(c_key, p_key, color="orange", title="PURCHASED")

# PLACED edges: Customer → Order
rows2 = cypher("""
    MATCH (c:Customer)-[:PLACED]->(o:SalesOrder)
    RETURN c.customer_id, o.order_id, o.total, o.status
""")

for row in rows2:
    c_id, o_id, o_total, o_status = row

    c_key = f"customer_{c_id}"
    if c_key not in added_nodes:
        net.add_node(c_key, label=f"Customer {c_id}", color="#e94560", shape="dot", size=25)
        added_nodes.add(c_key)

    o_key = f"order_{o_id}"
    if o_key not in added_nodes:
        net.add_node(o_key, label=f"Order {o_id}\n${o_total}", color="#533483", shape="square", size=12)
        added_nodes.add(o_key)

    net.add_edge(c_key, o_key, color="orange", title="PLACED")

net.save_graph("ecommerce_graph.html")
IFrame("ecommerce_graph.html", width="100%", height=620)

## Cleanup (optional)

In [ ]:
# ── Step 10 │ Cleanup ─────────────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────────
# Clean up — delete the instance to free the wrapper pod memory
try:
    delete(f"/api/instances/{instance_id}")
    print(f"Instance {instance_id} deleted — wrapper pod will be removed")
except Exception as e:
    print(f"Instance {instance_id} already gone or not found: {e}")